# Lire le livrable « classification du stationnement »

Ce carnet se lit de haut en bas et se rejoue tel quel, sans accès au laboratoire : il n'a besoin que du dossier `livrable/` (le zip téléchargé depuis l'onglet **Livrable**, une fois décompressé).

**Ce que contient le livrable** : 8 modèles, un par groupe (panoramique ou non, côté droit ou gauche, tâche « nombre de places » ou « forme de la place »). Pour chacun : le modèle (`.pt`), la liste de ses images par lot (`_lots.csv`), ses mesures (`_mesures.json`) et sa matrice de confusion (`_confusion_validation.csv`).

**Ce que fait le carnet**
1. il présente les 8 modèles et leurs chiffres ;
2. il apprend à les lire (rappel, hasard, réponse triviale) en les **recalculant** depuis la matrice de confusion ;
3. il **contrôle** le livrable (empreinte des modèles, aucune collection d'images dans deux lots) ;
4. il permet d'**essayer** un modèle sur vos propres images (étape facultative, demande `ultralytics`).

> **À retenir avant de commencer** : toutes les mesures portent sur le lot de **validation**. Le lot de **test** est scellé : il n'a pas été ouvert, aucun chiffre de test n'existe.

## 0. Préparation

In [ ]:
import csv, hashlib, json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# Où est le dossier « livrable » ? On regarde à côté du carnet, puis dans le dossier courant.
# Sinon, indiquer le chemin à la main : DOSSIER = Path("/chemin/vers/livrable")
DOSSIER = next((p for p in (Path("livrable"), Path("."), Path("..") / "livrable") if (p / "synthese.csv").is_file()), None)
if DOSSIER is None:
    raise SystemExit("synthese.csv introuvable : décompressez livrable.zip à côté de ce carnet, ou fixez DOSSIER ci-dessus.")
print("Livrable lu dans :", DOSSIER.resolve())
print(len(list(DOSSIER.iterdir())), "fichiers")

## 1. Les 8 modèles

Une ligne par groupe. Les colonnes importantes :

| Colonne | Sens |
|---|---|
| `rappel_moyen` | moyenne des rappels de chaque catégorie, **chaque catégorie comptant autant**. C'est la mesure principale. Celle-ci est pour le fichier du modèle livré (graine 0). |
| `rappel_moyen_3_graines` ± `ecart_3_graines` | la même mesure moyennée sur trois entraînements (graines 0, 1, 2). **C'est ce chiffre qu'il faut retenir** : un seul entraînement peut varier de plusieurs points. |
| `hasard` | le rappel moyen d'un modèle qui répondrait au hasard (1 ÷ nombre de catégories). |
| `exactitude` / `reponse_triviale` | part d'images bien classées, et ce qu'obtiendrait « toujours la catégorie la plus fréquente ». |
| `train` / `val` / `test` | nombre d'images par lot. |

In [ ]:
synthese = pd.read_csv(DOSSIER / "synthese.csv").set_index("groupe")
synthese[["modele", "essai_livre", "train", "val", "test", "rappel_moyen_3_graines", "ecart_3_graines", "rappel_moyen", "hasard", "exactitude", "reponse_triviale"]]

Pour voir d'un coup d'œil si un modèle apporte quelque chose : son rappel moyen (3 graines) doit être **nettement au-dessus du hasard**, plus loin que le trait d'incertitude (l'écart entre graines).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = range(len(synthese))
ax.barh(list(y), synthese["rappel_moyen_3_graines"], xerr=synthese["ecart_3_graines"], color="#2354d6", label="rappel moyen (3 graines)", capsize=3)
ax.scatter(synthese["hasard"], list(y), color="#c0392b", zorder=3, marker="|", s=400, label="hasard")
ax.set_yticks(list(y), [g.replace("_", " ") for g in synthese.index])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("rappel moyen sur le lot de validation")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Comment lire ce graphique.** Plus la barre dépasse le trait rouge, mieux le modèle distingue les catégories. Les groupes « forme » reposent sur très peu d'images de validation (5 à 13) : leurs chiffres sont **indicatifs seulement** — une image de plus déplace le résultat de plusieurs points.

## 2. Un groupe en détail

Choisissez un groupe en changeant la valeur de `GROUPE`, puis rejouez les cellules suivantes.

In [ ]:
GROUPE = "panoramique_droite_nombre"   # ← à changer : voir la liste ci-dessus
assert GROUPE in synthese.index, f"groupe inconnu, choisir parmi : {list(synthese.index)}"
mesures = json.loads((DOSSIER / f"{GROUPE}_mesures.json").read_text(encoding="utf-8"))
val = mesures["validation"]
classes = val["classes"]
print(f"{GROUPE} : modèle {mesures['modele']}, essai {mesures['essai_livre']} (graine {mesures['graine']})")
print("catégories :", classes)
print("règle de choix du modèle :", mesures["regle_de_choix"])
print("lot de test :", mesures["test"])

### La matrice de confusion

Lignes : la **vraie** catégorie. Colonnes : celle que le modèle a **prédite**. La diagonale, ce sont les bonnes réponses ; tout le reste, ce que le modèle confond avec quoi.

In [ ]:
confusion = pd.DataFrame(val["confusion"], index=[f"vrai {c}" for c in classes], columns=[f"prédit {c}" for c in classes])
fig, ax = plt.subplots(figsize=(4.5 + len(classes) * 0.4, 3.8 + len(classes) * 0.3))
ax.imshow(confusion.values, cmap="Blues")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, confusion.values[i, j], ha="center", va="center", color="white" if confusion.values[i, j] > confusion.values.max() / 2 else "black")
ax.set_xticks(range(len(classes)), confusion.columns)
ax.set_yticks(range(len(classes)), confusion.index)
ax.set_title(GROUPE.replace("_", " "))
plt.tight_layout()
plt.show()

### Recalculer les mesures soi-même

Rien n'est à croire sur parole : tous les chiffres se retrouvent depuis la matrice.

- **rappel d'une catégorie** = bonnes réponses de la ligne ÷ total de la ligne (parmi les vraies images de cette catégorie, la part retrouvée) ;
- **précision d'une catégorie** = bonnes réponses de la colonne ÷ total de la colonne (parmi les images rangées dans cette catégorie, la part qui y était vraiment) ;
- **rappel moyen** = moyenne des rappels ;
- **exactitude** = diagonale ÷ total ;
- **réponse triviale** = effectif de la plus grosse catégorie ÷ total.

Une catégorie de **moins de 20 images** est marquée « non fiable » : son rappel est indicatif.

In [ ]:
m = confusion.values
lignes = m.sum(axis=1)
colonnes = m.sum(axis=0)
tableau = pd.DataFrame({
    "images": lignes,
    "rappel": [m[i, i] / lignes[i] if lignes[i] else float("nan") for i in range(len(classes))],
    "précision": [m[i, i] / colonnes[i] if colonnes[i] else float("nan") for i in range(len(classes))],
    "fiable (≥ 20 images)": ["oui" if n >= 20 else "NON" for n in lignes],
}, index=classes)
display(tableau.round(3))

rappel_moyen = tableau["rappel"].mean()
exactitude = m.trace() / m.sum()
triviale = lignes.max() / m.sum()
hasard = 1 / len(classes)
print(f"rappel moyen  : {rappel_moyen:.3f}   (fichier : {val['rappel_moyen']:.3f})")
print(f"exactitude    : {exactitude:.3f}   (fichier : {val['exactitude']:.3f})")
print(f"réponse triviale : {triviale:.3f}   (fichier : {val['reponse_triviale_exactitude']:.3f})")
print(f"hasard        : {hasard:.3f}")
assert abs(rappel_moyen - val["rappel_moyen"]) < 1e-9 and abs(exactitude - val["exactitude"]) < 1e-9, "écart entre le recalcul et le fichier !"
print("→ le recalcul correspond aux mesures du fichier.")

### Ce que disent ces chiffres

- **Rappel moyen contre hasard** : s'ils sont proches, le modèle ne distingue presque rien.
- **Exactitude contre réponse triviale** : une exactitude qui ne dépasse pas la réponse triviale n'apporte rien. Un modèle qui répondrait toujours « la catégorie la plus fréquente » aurait déjà ce score.
- **Une seule graine** : le fichier du modèle est celui de la graine 0. La moyenne sur 3 graines est donnée ci-dessous ; **un écart plus petit que l'écart entre graines est du bruit**.

In [ ]:
g = mesures["trois_graines"]
print("essais des trois graines :", g["essais"])
print(f"rappel moyen sur 3 graines : {g['rappel_moyen_moyenne']:.3f} ± {g['rappel_moyen_ecart']:.3f}")
print(f"exactitude moyenne          : {g['exactitude_moyenne']:.3f}")
print(f"rappel moyen du fichier livré (graine 0) : {val['rappel_moyen']:.3f}")

## 3. Contrôler le livrable

Deux vérifications qu'on peut refaire à tout moment.

**a) Le modèle est bien celui qui a été mesuré** : l'empreinte du fichier `.pt` doit correspondre à celle des mesures.

In [ ]:
lignes = []
for groupe in synthese.index:
    mes = json.loads((DOSSIER / f"{groupe}_mesures.json").read_text(encoding="utf-8"))
    empreinte = hashlib.sha256((DOSSIER / f"{groupe}.pt").read_bytes()).hexdigest()
    lignes.append({"groupe": groupe, "empreinte conforme": empreinte == mes["poids_sha256"]})
controle = pd.DataFrame(lignes).set_index("groupe")
display(controle)
assert controle["empreinte conforme"].all(), "un modèle ne correspond pas à ses mesures"

**b) Pas de fuite entre les lots.** Des images voisines (même rue, même enregistrement) se ressemblent : si l'une est en entraînement et l'autre en validation, le modèle « triche » et la mesure est trop belle. Le découpage se fait donc **par collection d'images**, jamais image par image. Contrôle : aucune collection ne doit apparaître dans deux lots.

In [ ]:
lignes = []
for groupe in synthese.index:
    lots = pd.read_csv(DOSSIER / f"{groupe}_lots.csv")
    par_collection = lots.groupby("groupe")["jeu"].nunique()
    lignes.append({
        "groupe": groupe,
        "images": len(lots),
        "train": int((lots["jeu"] == "train").sum()), "val": int((lots["jeu"] == "val").sum()), "test": int((lots["jeu"] == "test").sum()),
        "collections": len(par_collection),
        "collections dans 2 lots": int((par_collection > 1).sum()),
    })
fuites = pd.DataFrame(lignes).set_index("groupe")
display(fuites)
assert (fuites["collections dans 2 lots"] == 0).all(), "une collection est présente dans deux lots"
print("→ aucune collection n'est partagée entre deux lots.")

Les images elles-mêmes ne sont pas dans le livrable (ce sont des photos de rue) : `_lots.csv` en donne la **liste** — fichier, lot, catégorie, collection, origine — pour retrouver chaque image.

Répartition des catégories par lot pour le groupe choisi :

In [ ]:
lots = pd.read_csv(DOSSIER / f"{GROUPE}_lots.csv")
pd.crosstab(lots["classe"], lots["jeu"])

## 4. Essayer un modèle sur vos images *(facultatif)*

Il faut `ultralytics` (`pip install ultralytics`). Le modèle attend des images **déjà recadrées comme celles du jeu** (le recadrage de la zone de stationnement, côté droit ou gauche selon le groupe), et ramenées à **64 × 64 pixels** — c'est la taille de l'entraînement. La cellule fait ce redimensionnement.

Déposez des images dans un dossier `mes_images/` à côté du carnet. Sans dossier, la cellule se contente de le dire.

In [ ]:
DOSSIER_IMAGES = Path("mes_images")

try:
    from PIL import Image
    from ultralytics import YOLO
except ImportError as e:
    print("Étape facultative sautée :", e, "→ pip install ultralytics")
else:
    images = sorted(p for p in DOSSIER_IMAGES.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if DOSSIER_IMAGES.is_dir() else []
    if not images:
        print(f"Aucune image trouvée dans {DOSSIER_IMAGES}/ : déposez-en quelques-unes puis rejouez la cellule.")
    else:
        modele = YOLO(str(DOSSIER / f"{GROUPE}.pt"))
        for chemin in images[:12]:
            image = Image.open(chemin).convert("RGB").resize((64, 64), Image.LANCZOS)
            res = modele.predict(image, imgsz=64, verbose=False)[0]
            probas = {res.names[i]: float(p) for i, p in enumerate(res.probs.data.tolist())}
            meilleur = max(probas, key=probas.get)
            print(f"{chemin.name:30s} → {meilleur}   " + "  ".join(f"{c}: {p:.2f}" for c, p in probas.items()))

Un résultat sur une image ne prouve rien : le rappel moyen ci-dessus est ce qui dit, **sur beaucoup d'images**, ce que vaut le modèle.

## 5. Ce qu'on affirme, ce qu'on n'affirme pas

**On affirme**
- les chiffres de ce carnet, mesurés sur le lot de validation, reproductibles depuis les fichiers du livrable ;
- que les lots sont séparés par collection (contrôle du §3) ;
- que le choix du modèle (yolo26n ou yolo26s) s'est fait sur la validation, par la moyenne de 3 graines.

**On n'affirme pas**
- de chiffre sur le lot de test : il est scellé, jamais ouvert. Un chiffre de test n'existera qu'une fois, à la fin, si on décide de l'ouvrir ;
- que les groupes « forme » sont fiables : 5 à 13 images de validation, les chiffres sont indicatifs ;
- que les modèles généralisent à d'autres villes ou d'autres prises de vue : les images viennent de peu d'enregistrements ;
- que le modèle égale un humain : l'accord entre deux annotateurs n'a pas encore été mesuré, il borne ce qu'on peut attendre.